# Microsoft Foundryతో ఒక మోడల్‌ను ఫైన్-ట్యూన్ చేయండి

ఈ నోట్‌బుక్ ప్రస్తుత [Microsoft Foundry ఫైన్-ట్యూనింగ్ వర్క్‌ఫ్లో](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst) ను అనుసరిస్తుంది. ఇది మీ Foundry వనరు యొక్క `/openai/v1/` ఎండ్‌పాయింట్‌కి సూచించిన **OpenAI Python SDK (v1)** ను ఉపయోగిస్తుంది, అందువల్ల అదే కోడ్ OpenAI ప్లాట్‌ఫామ్ఖე కూడా కేవలం క్లయింట్ సెటప్ మాత్రమే మార్చి నడుస్తుంది.

> **అవసరాలు**
> - ఒక [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) ప్రాజెక్ట్, **Foundry Owner** పాత్రతో (ఫైన్-ట్యూన్ మరియు డిప్లాయ్ చేయడానికి అవసరము).
> - `pip install "openai>=1.0" python-dotenv`
> - `AZURE_OPENAI_ENDPOINT` మరియు `AZURE_OPENAI_API_KEY` ఉన్న `.env` ఫైల్ (చూడండి [కోర్సు సెటప్ గైడ్](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)).
> - ప్రస్తుతం మద్దతు ఉన్న బేస్ మోడల్, ఉదాహరణకు `gpt-4.1-mini` (చూడండి [ఫైన్-ట్యూనింగ్ మోడల్స్ జాబితా](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)).

ఫైన్-ట్యూనింగ్ అంటే ఒక ఫౌండేషన్ మోడల్‌ను మీ పనికి సంబంధించిన అదనపు ఉదాహరణలపై మళ్ళీ శిక్షణ ఇవ్వడం ద్వారా మెరుగుపరచడం. ప్రాంప్ట్ ఇంజనీరింగ్ సాంకేతికతలు _few-shot learning_ మరియు _retrieval-augmented generation_ వంటి నివేదికలకు సంబంధించిన డేటా తో ప్రాంప్ట్‌ను మెరుగుపరుస్తాయి, కానీ అవి మోడల్ కాంటెక్స్ట్ విండో పరిమితులు కలిగి ఉంటాయి.

ఫైన్-ట్యూనింగ్ ద్వారా మీరు మోడల్‌ను నేరుగా మళ్ళీ శిక్షణ ఇస్తారు (కాంటెక్స్ట్ విండోలో సరిపోని చాలా ఉదాహరణలు ఉపయోగించి) మరియు ఆపై _కస్టమ్_ వెర్షన్‌ని అందిస్తారు, ఇది ఇన్ఫెరెన్స్ సమయంలో ఆ ఉదాహరణలను అవసరం పెట్టదు. ఇది నాణ్యతను మెరుగుపరిచే, కాంటెక్స్ట్ విండోను ఖాళీ చేయడంలో సహాయపడుతుంది, అలాగే ప్రాంప్ట్‌ల పరిమాణం తగ్గించి ఖర్చు, లేటెన్సీని తగ్గించవచ్చు. Foundry లోపల **LoRA (లో-రాంక్ అనుకూలత)** ని సమర్థవంతమైన శిక్షణకు ఉపయోగిస్తుంది.

Foundry మూడు సాంకేతికతలను మద్దతు ఇస్తుంది: **సూపర్వైజ్డ్ ఫైన్-ట్యూనింగ్ (SFT)** - ఇందులో ఈ ట్యుటోరియల్ ఉపయోగిస్తున్నది - మరియు మరో రెండు **DPO** (ప్రాధాన్యత సదృశ్యం) మరియు **RFT** (రిఇన్ఫోర్స్‌మెంట్ ఫైన్-ట్యూనింగ్). వర్క్‌ఫ్లో నాలుగు అడుగులు ఉన్నాయి:

1. మీ శిక్షణ మరియు ధృవీకరణ డేటాను సిద్ధం చేసి అప్‌లోడ్ చేయండి.
2. ఫైన్-ట్యూనింగ్ మోడల్ సృష్టించడానికి శిక్షణ జాబ్ నడపండి.
3. జాబ్‌ను మానిటర్ చేసి, మెట్రిక్స్‌ను సమీక్షించి, ఒక చెక్పాయింట్ ఎంచుకోండి.
4. ఫైన్-ట్యూన్ చేసిన మోడల్‌ను డిప్లాయ్ చేసి, దీన్ని ఇన్ఫెరెన్స్ కోసం ఉపయోగించండి.

ఈ ట్యుటోరియలులో మేము `gpt-4.1-mini` ని SFT తో ఫైన్-ట్యూన్ చేస్తూ, పీరియాడిక్ టేబుల్ గురించి ప్రశ్నలకు లిమరిక్ రూపంలో జవాబు చెప్పే చాట్బాట్ ను తయారు చేస్తున్నాము.

---


### దశ 1.1: మీ డాటాసెట్ ను సిద్ధం చేయండి

ఒక ఎలిమెంట్ గురించి ప్రశ్నలకు లిమరిక్ తో జవాబులు చెప్పడం ద్వారా మీరు మూలకాల వారీ పట్టికను అర్థం చేసుకోవడంలో సహాయపడే చాట్‌బాట్ ను తయారు చేద్దాం. _ఈ_ సులభమైన పాఠంలో, మేము కేవలం మోడల్‌ను శిక్షించడానికి కొన్ని నమూనా ప్రతిస్పందనలతో డాటాసెట్‌ను సృష్టిస్తాము, ఇవి డాటా యొక్క ఆశించబడిన ఫార్మాట్ ను చూపిస్తాయి. వాస్తవ ప్రపంచంలో మీరు ఇంకా ఎక్కువ ఉదాహరణలతో డాటాసెట్‌ను రూపొందించుకోవాలి. ఒకటి ఉంటే మీరు ఓపెన్ డాటాసెట్‌ను (మీ అప్లికేషన్ డొமைన్ కోసం) కూడా ఉపయోగించవచ్చు, మరియు దానికి సరిపడే విధంగా దాన్ని మళ్ళీ ఫార్మాట్ చేయవచ్చు.

మేము `gpt-4.1-mini` మీద దృష్టి సారించి ఒకే టర్న్ రిస్పాన్స్ (చాట్ కంప్లీషన్) కోసం [ఈ సూచించిన ఫార్మాట్](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) ద్వారా ఉదాహరణలను సృష్టించవచ్చు, ఇది OpenAI చాట్ కంప్లీషన్ అవసరాలని ప్రతిబింబిస్తుంది. మీరు పలురు-టర్న్ సంభాషణాత్మక కంటెంట్ బత్యదిస్తే, [పలురు-టర్న్ ఉదాహరణ ఫార్మాట్](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) ఉపయోగిస్తారు, దీనిలో ఫైన్-ట్యూనింగ్ ప్రక్రియలో ఉపయోగించవలసిన సందేశాలను (లేదా ఉపయోగించకూడదు అన్న దానిని) సూచించడానికి `weight` పారామీటర్ ఉంటుంది.

ఇక్కడ మా పాఠం కోసం మేము తేలికైన ఒకే టర్న్ ఫార్మాట్ ఉపయోగిస్తాము. డేటా [jsonl ఫార్మాట్](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst) లో ఉంటుంది, ప్రతి లైన్ కి 1 రికార్డు ఉంటుంది, ఇది JSON-ఫార్మాట్ ఆబ్జెక్టుగా ప్రాతినిధ్యం వహిస్తుంది. క్రింద స్నిపెట్ 2 రికార్డులను నమూనాగా చూపిస్తుంది - మా పూర్తి నమూనా సెట్ కోసం [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) చూడండి (10 ఉదాహరణలు) ఇది మా ఫైన్-ట్యూనింగ్ పాఠం కోసం ఉపయోగిస్తాము. **గమనిక:** ప్రతి రికార్డు _ఒకే లైన్‌లో_ నిర్వచించబడాలి (సాధారణంగా ఫార్మాట్ చేయబడిన JSON ఫైల్ లో లైన్లుగా విడగొట్టరు)

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

వాస్తవ ప్రపంచంలో మంచి ఫలితాల కోసం మీరు చాలా పెద్ద ఉదాహరణల సెట్ అవసరమవుతుంది - డేటా యొక్క నాణ్యత మరియు ఫైన్-ట్యూనింగ్ కోసం సమయం/ఖర్చుల మధ్య వ్యత్యాసం ఉంటుంది. మేము ఈ చిన్న సెట్ ఉపయోగిస్తున్నాం కాబట్టి మేము ఫైన్-ట్యూనింగ్ ప్రక్రియను త్వరగా పూర్తి చేసి ప్రదర్శించగలుగుతాము. మరింత క్లిష్టమైన ఫైన్-ట్యూనింగ్ పాఠం కోసం [ఈ OpenAI కుక్‌బుక్ ఉదాహరణ](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) చూడండి.


---

### దశ 1.2: మీ డేటాసెట్‌ను అప్‌లోడ్ చేయండి

ఫైళ్ళ API (`purpose="fine-tune"`) తో మీ శిక్షణ మరియు ధ్రువీకరణ ఫైళ్లను Foundryకి అప్‌లోడ్ చేయండి. **ధ్రువీకరణ ఫైల్** ఇవ్వడం ద్వారా Foundry ధ్రువీకరణ నష్టం నివేదికను అందిస్తుంది, తద్వారా మీరు ఒవర్‌ఫిటింగ్‌ను గుర్తించవచ్చు.

క్రింద ఇవ్వబడిన కో드를 నడపడంలో ముందు, మీరు ఖాతరించుకున్నది:
 - SDKని ఇన్‌స్టాల్ చేసుకోండి: `pip install "openai>=1.0" python-dotenv`
 - `AZURE_OPENAI_ENDPOINT` మరియు `AZURE_OPENAI_API_KEY`తో `.env` ఫైల్ తయారు చేయండి

కోడ్ Foundry వనరుల `/openai/v1/` ఎండ్‌పాయింట్ సూచించే OpenAI క్లయింట్ సృష్టిస్తుంది, అప్పుడు రెండు ఫైళ్లను అప్‌లోడ్ చేసి వాటి IDలను ప్రింట్ చేస్తుంది.


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### దశ 2.1: SDK ఉపయోగించి ఫైన్-ట్యూనింగ్ జాబ్ సృష్టించండి


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### దశ 2.2: ఉద్యోగ స్థితిని తనిఖీ చేయండి

`client.fine_tuning.jobs` API తో మీరు చేయగల కొన్ని విషయాలు ఇవి:
- `client.fine_tuning.jobs.list(limit=<n>)` - చివరి n ఫైన్-ట్యూనింగ్ ఉద్యోగాలను జాబితా చేయండి
- `client.fine_tuning.jobs.retrieve(<job_id>)` - ఒక నిర్దిష్ట ఉద్యోగ విపరాలు పొందండి
- `client.fine_tuning.jobs.cancel(<job_id>)` - ఒక ఉద్యోగాన్ని రద్దు చేయండి
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - ఉద్యోగం నుండి ఈవెంట్ల జాబితాను పొందండి
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - డిప్లాయబుల్ చెక్‌పాయింట్లను జాబితా చేయండి (చివరి కొన్ని యెక్కులు)

ఉద్యోగం ప్రారంభమైనప్పుడు, Foundry మొదట _శిక్షణ ఫైల్ ని ధృవీకరిస్తుంది_ డేటా సరైన ఫార్మాట్ లో ఉందో లేదో చూసుకోవడానికి. ఆపై శిక్షణ నడుస్తుంది, మరియు మోడల్ మరియు డేటాసెట్ పరిమాణం మీద ఆధారపడి కావలసిన సమయం క్షణాల నుండి గంటల వరకు ఉండవచ్చు.


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### దశ 2.3: పురోగతిని పర్యవేక్షించడానికి ఈవెంట్లను ట్రాక్ చేయండి


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### దశ 2.4: Foundry పోర్టల్‌లో స్టాటస్, మెట్రిక్‌లు, మరియు checkpoints సమీక్షించండి


మీరు [Microsoft Foundry పోర్టల్](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) లో **Build > Fine-tuning** కింద మీ జాబ్‌ను కూడా ట్రాక్ చేయవచ్చు. దాని స్థితి, శిక్షణ సంఘటనలు, హైపర్‌పరామితులు, మరియు ప్రమాణాలను చూడడానికి మీ జాబ్‌ను ఎంచుకోండి. ఈ ప్రమాణాలను గమనించండి:

- `train_loss` మరియు `full_valid_loss` - కాలక్రమేణా తగ్గాలి.
- `train_mean_token_accuracy` మరియు `full_valid_mean_token_accuracy` - పెరిగాలి.

శిక్షణ మరియు ధృవీకరణ వంగులు విభిన్నంగా ఉంటే, మీరు ఓవర్‌ఫిటింగ్ అవ్వచ్చు - తక్కువ ఎపోక్స్ లేదా చిన్న లెర్నింగ్ రేటు మల్టిప్లయర్ని ప్రయత్నించండి. దిగువ చిత్రణ మీరు చూడబోయే స్థితి, సందేశం, మరియు మేట్రిక్ ప్యానెల్‌లను చూపిస్తుంది (ఖచ్చిత UI ప్రొవైడర్‌పై ఆధారపడి మారవచ్చు).

![Fine-tuning job status](../../../../../translated_images/te/fine-tuned-model-status.563271727bf7bfba.webp)


మీరు కింది విధంగా చూపినట్లు విజువల్ డాష్‌బోర్డులో మరింతకి స్క్రోల్ చేస్తూ స్థితి సందేశాలు మరియు ప్రమాణాలను కూడా వీక్షించవచ్చు:

| సందేశాలు | ప్రమాణాలు |
|:---|:---|
| ![Messages](../../../../../translated_images/te/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/te/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### దశ 3.1: సరికొత్త ఫైన్-ట్యూన్ చేసిన మోడల్ id ను తీసుకోండి

పని విజయవంతమైనప్పుడు, `response.fine_tuned_model` లో మీ అనుకూల మోడల్ id ఉంటుంది (ఉదాహరణకు, `gpt-4.1-mini-2025-04-14.ft-...`). Azure లో మీరు ఆ మోడల్ **ను అమలు** చేసేందుకు ముందు డిప్లాయ్ చేయాలి.


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### దశ 3.2: ఫైన్-ట్యూన్ చేసిన మోడల్‌ను డిప్లాయ్ చేయడం

OpenAI ప్లాట్‌ఫామ్ (అక్కడ మీరు ఫైన్-ట్యూన్ చేసిన మోడల్ id ను నేరుగా పిలవవచ్చు) కన్నా భిన్నంగా, Microsoft Foundry మీరు ముందుగా **మోడల్‌ను డిప్లాయ్ చేయాలని** కోరుతుంది. సులభమైన విధానం [Foundry పోర్టల్](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) కు వెళ్ళడం: **Build > Fine-tuning** లోకి వెళ్లి, పూర్తి అయిన జాబ్‌ను ఎంచుకోండి, మరియు **Deploy** ఆప్షన్ ఎంచుకోండి. డిప్లాయ్‌మెంట్‌కు పేరు ఇవ్వండి (ఉదాహరణకు, `elements-limerick`) - ఆ డిప్లాయ్‌మెంట్ పేరు కోడ్ నుండి పిలవడంలో ఉపయోగిస్తారు.

మీరు నియంత్రణ-ప్లేన్ REST/ARM API ద్వారా కూడా ప్రోగ్రామికల్‌గా డిప్లాయ్ చేయవచ్చు; [deployment guide](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst) చూడండి. ఒకసారి డిప్లాయ్ చేసిన కస్టమ్ మోడల్ గంటలవారీగా బిల్లింగ్ చేస్తుంది, మరియు క్రియాశీలత లేని డిప్లాయ్‌మెంట్ 15 రోజుల తర్వాత తొలగించబడుతుంది అని గమనించండి.


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### దశ 3.3: ఫౌండ్రీ ప్లేగ్రౌండ్‌లో మీ ఫైన్-ట్యూన్ చేసిన మోడల్ ను పరీక్షించండి

మీరు [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **Playground** లో కూడా నియోజకవర్గం చేసిన మోడల్ ను పరీక్షించవచ్చు - మోడల్ డ్రాప్‌డౌన్ నుండి మీ ఫైన్-ట్యూన్ చేసిన నియోజకవర్గం ని ఎంచుకుని ప్రాంప్ట్ ని ప్రయత్నించండి. మీరు శిక్షణ పొందిన **అనే సిస్టమ్ సందేశాన్ని** ఉపయోగించండి; వేరే సిస్టమ్ సందేశం ప్రవర్తనను మార్చవచ్చు.

> **సూచన:** ఫైన్-ట్యూన్డ్ మోడల్ ను బేస్ `gpt-4.1-mini` తో పక్కకు పక్కగా పోల్చండి. మీరు ఇచ్చిన ఉదాహరణల నుండి ఫైన్-ట్యూన్ చేయబడిన మోడల్ లిమరిక్ ఫార్మాట్ లో సమాధానాలు ఇస్తుండగా, బేస్ మోడల్ కేవలం సిస్టమ్ ప్రాంప్ట్ ని అనుసరిస్తుందని గమనించండి.

**ఇది ప్రక్రియను చూపించడానికి ఉద్దేశించిన సాధారణ ఉదాహరణ మాత్రమే, ఇది ఒక వాస్తవ ప్రపంచ డేటాసెట్ కాకపోవచ్చు.** ప్రొడక్షన్‌లో మీరు వాస్తవ డేటా (ఉదాహరణకు, కస్టమర్ సర్వీస్ కోసం ఉత్పత్తి క్యాటలాగ్) పై ఫైన్-ట్యూన్ చేస్తారు, అక్కడ తగిన శ్రేణి తేడా చాలా స్పష్టంగా ఉంటుంది - మరియు ఆశక్తిని ప్రాంప్ట్ ఇంజనీరింగ్ ద్వారా మాత్రమే సరిపోల్చడం ప్రతి కాల్ కి మరిన్ని టోకెన్ల ఖర్చును కలిగిస్తుంది. మరింత పూర్తిగా గాఇడ్ కోసం, [OpenAI Cookbook fine-tuning guide](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) మరియు [Foundry fine-tuning tutorial](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst) చూడండి.

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
